# 04 - State Machine de Orquestração

Este notebook implementa e valida a máquina de estados do projeto **Agro Leads Orchestrator**.

A proposta é controlar o ciclo de vida operacional dos leads agrícolas, evitando contatos duplicados, reduzindo atrito com clientes e priorizando oportunidades comerciais reais.

## Objetivos

Neste notebook serão testadas as principais regras de negócio:

- busca de próximos leads para robô;
- aplicação de cooldown de 48 horas;
- bloqueio de robôs durante cooldown;
- resposta de WhatsApp gerando fila prioritária;
- transferência assistida do robô para vendedor humano;
- conversão do lead após venda;
- histórico auditável de eventos;
- resumo de status da base.

In [11]:
from pathlib import Path
import sqlite3
import sys
from src.orquestrador import OrquestradorOmnichannelLeads

import pandas as pd

In [12]:
#Localizar raiz do projeto

RAIZ_PROJETO = Path.cwd()

if RAIZ_PROJETO.name == "notebooks":
    RAIZ_PROJETO = RAIZ_PROJETO.parent

if str(RAIZ_PROJETO) not in sys.path:
    sys.path.append(str(RAIZ_PROJETO))

print("Raiz do projeto:", RAIZ_PROJETO)

Raiz do projeto: c:\Users\DataCore\Desktop\projetos_Git\agro-leads-orchestrator


In [14]:
#Caminho do projeto

CAMINHO_BANCO = RAIZ_PROJETO / "dados" / "agro_leads.db"

print("Banco:", CAMINHO_BANCO)
print("Banco existe?", CAMINHO_BANCO.exists())

Banco: c:\Users\DataCore\Desktop\projetos_Git\agro-leads-orchestrator\dados\agro_leads.db
Banco existe? True


In [15]:
#Criar instância do orquestrador

orquestrador = OrquestradorOmnichannelLeads(CAMINHO_BANCO)

print("Orquestrador inicializado com sucesso.")

Orquestrador inicializado com sucesso.


In [16]:
#Resumo por status antes das operações

resumo_status_inicial = orquestrador.obter_resumo_status()

resumo_status_inicial

,status_atual,quantidade,score_medio
0,Disponível,683811,59.64
1,Em Cooldown,114146,8.95
2,Fila Prioritária,57024,107.59
3,Convertido,56814,0.00
4,Em Atendimento,38205,17.89


In [17]:
#Recalcular score dinâmico

orquestrador.calcular_score_prioridade()

print("Score de prioridade recalculado com sucesso.")

Score de prioridade recalculado com sucesso.


In [18]:
#Próximos leads para robô

proximos_leads_robo = orquestrador.obter_proximos_leads_robo(
    limite=10
)

proximos_leads_robo

,id_cliente,nome,telefone,cultura,estagio_atual,status_atual,ultimo_contato,cooldown_ate,score_prioridade
0,149,Igor Martins,5564900000149,Cana,Plantio,Disponível,NaN,None,163.35
1,2099,Paulo Martins,5515900002099,Cana,Plantio,Disponível,2026-06-05 06:37:57,None,163.35
2,2449,José Santos,5566900002449,Cana,Plantio,Disponível,2026-06-13 10:49:57,None,163.35
3,3049,João Pereira,5534900003049,Cana,Plantio,Disponível,2026-06-08 00:34:57,None,163.35
4,3249,Lucas Costa,5567900003249,Cana,Plantio,Disponível,2026-06-26 16:38:57,None,163.35
5,3349,Carlos Carvalho,5567900003349,Cana,Plantio,Disponível,2026-06-02 01:46:57,None,163.35
6,3499,Henrique Costa,5567900003499,Cana,Plantio,Disponível,2026-06-19 16:36:57,None,163.35
7,3949,Henrique Oliveira,5513900003949,Cana,Plantio,Disponível,2026-06-22 21:10:57,None,163.35
8,6449,João Rodrigues,5516900006449,Cana,Plantio,Disponível,2026-06-06 20:01:57,None,163.35
9,6849,Gustavo Costa,5544900006849,Cana,Plantio,Disponível,2026-06-27 11:05:57,None,163.35


In [19]:
#Selecionar lead de teste

id_lead_teste = int(proximos_leads_robo.iloc[0]["id_cliente"])

print("Lead selecionado para teste:", id_lead_teste)

orquestrador.obter_lead_por_id(id_lead_teste)

Lead selecionado para teste: 149


{'id_cliente': 149,
 'nome': 'Igor Martins',
 'telefone': '5564900000149',
 'cultura': 'Cana',
 'estagio_atual': 'Plantio',
 'status_atual': 'Disponível',
 'ultimo_contato': None,
 'cooldown_ate': None,
 'score_prioridade': 163.35}

In [20]:
#Não atendido e cooldown / Registrar evento não atendido

orquestrador.registrar_evento_nao_atendido(
    id_cliente=id_lead_teste,
    canal="Robô"
)

print("Evento de não atendido registrado.")

Evento de não atendido registrado.


In [21]:
#Conferir status após cooldown

orquestrador.obter_lead_por_id(id_lead_teste)

{'id_cliente': 149,
 'nome': 'Igor Martins',
 'telefone': '5564900000149',
 'cultura': 'Cana',
 'estagio_atual': 'Plantio',
 'status_atual': 'Em Cooldown',
 'ultimo_contato': '2026-06-30 21:48:30',
 'cooldown_ate': '2026-07-02 21:48:30',
 'score_prioridade': 24.5}

In [ ]:
#Verificar se o robô ainda consegue pegar esse lead
#Resultado esperado: False

novos_leads_robo = orquestrador.obter_proximos_leads_robo(
    limite=50
)

id_lead_teste in novos_leads_robo["id_cliente"].tolist()

False

In [23]:
#Registrar resposta de WhatsApp

orquestrador.registrar_resposta_whatsapp(
    id_cliente=id_lead_teste
)

print("Resposta de WhatsApp registrada.")

Resposta de WhatsApp registrada.


In [24]:
#Verificar status após WhatsApp

orquestrador.obter_lead_por_id(id_lead_teste)

{'id_cliente': 149,
 'nome': 'Igor Martins',
 'telefone': '5564900000149',
 'cultura': 'Cana',
 'estagio_atual': 'Plantio',
 'status_atual': 'Fila Prioritária',
 'ultimo_contato': '2026-06-30 21:53:50',
 'cooldown_ate': None,
 'score_prioridade': 250.0}

In [25]:
#Verificar fila humana

proximos_leads_humano = orquestrador.obter_proximos_leads_humano(
    limite=10
)

proximos_leads_humano

,id_cliente,nome,telefone,cultura,estagio_atual,status_atual,ultimo_contato,cooldown_ate,score_prioridade
0,38,Ricardo Souza,5544900000038,Cana,Plantio,Fila Prioritária,2026-06-21 09:32:57,None,250.0
1,95,Roberto Lima,5543900000095,Soja,Plantio,Fila Prioritária,2026-06-28 01:12:57,None,250.0
2,149,Igor Martins,5564900000149,Cana,Plantio,Fila Prioritária,2026-06-30 21:53:50,None,250.0
3,447,Daniel Costa,5535900000447,Soja,Plantio,Fila Prioritária,2026-06-08 17:10:57,None,250.0
4,539,Henrique Souza,5562900000539,Cana,Plantio,Fila Prioritária,2026-06-21 11:45:57,None,250.0
5,694,Igor Silva,5566900000694,Cana,Plantio,Fila Prioritária,2026-06-24 07:15:57,None,250.0
6,744,Henrique Almeida,5534900000744,Cana,Plantio,Fila Prioritária,NaN,None,250.0
7,2145,Luiz Fernandes,5519900002145,Cana,Plantio,Fila Prioritária,2026-05-30 20:58:57,None,250.0
8,2599,Marcos Nascimento,5511900002599,Cana,Plantio,Fila Prioritária,NaN,None,250.0
9,2644,Carlos Souza,5519900002644,Soja,Plantio,Fila Prioritária,NaN,None,250.0


In [26]:
#Confirmar que o lead está na fila humana
#resultado esperado: True

id_lead_teste in proximos_leads_humano["id_cliente"].tolist()

True

In [27]:
#Transferência assistida / Selecionar novo lead para teste de atendimento do robô

lead_para_transferencia = orquestrador.obter_proximos_leads_robo(
    limite=5
).iloc[0]

id_lead_transferencia = int(lead_para_transferencia["id_cliente"])

print("Lead para transferência assistida:", id_lead_transferencia)

orquestrador.obter_lead_por_id(id_lead_transferencia)

Lead para transferência assistida: 2099


{'id_cliente': 2099,
 'nome': 'Paulo Martins',
 'telefone': '5515900002099',
 'cultura': 'Cana',
 'estagio_atual': 'Plantio',
 'status_atual': 'Disponível',
 'ultimo_contato': '2026-06-05 06:37:57',
 'cooldown_ate': None,
 'score_prioridade': 163.35}

In [28]:
#Registrar atendimento do robô com transferência assistida

orquestrador.registrar_atendimento_robo_atendido(
    id_cliente=id_lead_transferencia
)

print("Transferência assistida registrada.")

Transferência assistida registrada.


In [29]:
#Verificar status após transferência

orquestrador.obter_lead_por_id(id_lead_transferencia)

{'id_cliente': 2099,
 'nome': 'Paulo Martins',
 'telefone': '5515900002099',
 'cultura': 'Cana',
 'estagio_atual': 'Plantio',
 'status_atual': 'Em Atendimento',
 'ultimo_contato': '2026-06-30 21:57:30',
 'cooldown_ate': None,
 'score_prioridade': 49.01}

In [30]:
#Venda realizada / Registrar sucesso de venda

orquestrador.registrar_sucesso_venda(
    id_cliente=id_lead_transferencia
)

print("Venda registrada com sucesso.")

Venda registrada com sucesso.


In [31]:
#Verificar status após venda

orquestrador.obter_lead_por_id(id_lead_transferencia)

{'id_cliente': 2099,
 'nome': 'Paulo Martins',
 'telefone': '5515900002099',
 'cultura': 'Cana',
 'estagio_atual': 'Plantio',
 'status_atual': 'Convertido',
 'ultimo_contato': '2026-06-30 21:58:41',
 'cooldown_ate': '2026-07-30 21:58:41',
 'score_prioridade': 0.0}

In [32]:
#Auditoria de eventos / Histórico do lead de cooldown e WhatsApp

orquestrador.obter_historico_cliente(id_lead_teste)

,id_evento,id_cliente,data_evento,canal,resultado,observacao
0,3,149,2026-06-30 21:53:50,WhatsApp,Resposta WhatsApp,Cliente respondeu ao bot e entrou na fila prio...
1,2,149,2026-06-30 21:48:30,Sistema,Cooldown Aplicado,Cooldown válido até 2026-07-02 21:48:30.
2,1,149,2026-06-30 21:48:30,Robô,Não Atendido,Lead colocado em cooldown por 48 horas.


In [33]:
#Histórico do lead de transferência e venda

orquestrador.obter_historico_cliente(id_lead_transferencia)

,id_evento,id_cliente,data_evento,canal,resultado,observacao
0,6,2099,2026-06-30 21:58:41,Humano,Venda,Venda realizada. Lead bloqueado até 2026-07-30...
1,5,2099,2026-06-30 21:57:30,Sistema,Transferência Assistida,Chamada transferida automaticamente para vende...
2,4,2099,2026-06-30 21:57:30,Robô,Atendido,Cliente atendeu ligação do robô.


In [34]:
#Resumo por status após operações

resumo_status_final = orquestrador.obter_resumo_status()

resumo_status_final

,status_atual,quantidade,score_medio
0,Disponível,797955,88.88
1,Fila Prioritária,57025,159.04
2,Convertido,56815,0.00
3,Em Atendimento,38205,26.62


In [35]:
#Comparação antes e depois

comparacao_status = resumo_status_inicial.merge(
    resumo_status_final,
    on="status_atual",
    how="outer",
    suffixes=("_antes", "_depois")
)

comparacao_status["variacao_quantidade"] = (
    comparacao_status["quantidade_depois"].fillna(0)
    - comparacao_status["quantidade_antes"].fillna(0)
)

comparacao_status

,status_atual,quantidade_antes,score_medio_antes,quantidade_depois,score_medio_depois,variacao_quantidade
0,Convertido,56814,0.00,56815.0,0.00,1.0
1,Disponível,683811,59.64,797955.0,88.88,114144.0
2,Em Atendimento,38205,17.89,38205.0,26.62,0.0
3,Em Cooldown,114146,8.95,NaN,NaN,-114146.0
4,Fila Prioritária,57024,107.59,57025.0,159.04,1.0


## Conclusões da State Machine

A máquina de estados demonstrou o controle operacional do ciclo de vida dos leads.

As principais regras validadas foram:

- leads não atendidos entram em cooldown por 48 horas;
- robôs deixam de acessar leads em cooldown;
- respostas de WhatsApp colocam o cliente em fila prioritária;
- robôs podem transferir chamadas atendidas para vendedores humanos;
- vendas alteram o status do lead para Convertido;
- leads convertidos ficam temporariamente fora da régua comercial;
- todos os eventos relevantes são registrados em histórico auditável.

Essa lógica reduz ligações duplicadas, melhora a experiência do cliente e aumenta a eficiência da equipe comercial.

In [36]:
#Fechar conexão

orquestrador.fechar_conexao()

print("Conexão encerrada.")

Conexão encerrada.
